# Module 0: Foundations Review

## Learning Objectives

By completing this notebook, you will:

1. **Perform** essential matrix operations (eigendecomposition, SVD) for factor analysis
2. **Implement** Principal Component Analysis from scratch and interpret results
3. **Apply** time series concepts including stationarity and autocovariance
4. **Prepare** for static and dynamic factor models in subsequent modules

## Prerequisites

- Undergraduate linear algebra (matrix multiplication, eigenvalues)
- Basic probability and statistics (mean, variance, covariance)
- Python programming fundamentals

## Estimated Time: 90-120 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "essential matrix operations (eigendecomposition, SVD) for factor analysis",
    "Principal Component Analysis from scratch and interpret results",
    "time series concepts including stationarity and autocovariance",
    "for static and dynamic factor models in subsequent modules"
])

## Setup and Imports

We'll use standard scientific computing libraries. All required packages should be available in a typical data science environment.

In [ ]:
# Purpose: Import required libraries for matrix operations and visualization
# Key Concept: NumPy for numerical computing, Matplotlib for visualization

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import linalg
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

---

# Part 1: Matrix Algebra Review

Factor models rely on matrix decompositions to extract latent structure from data. We'll review eigendecomposition and SVD, the two fundamental tools.

## 1.1 Eigendecomposition

For a symmetric matrix $A$, eigendecomposition finds:
$$A = V \Lambda V'$$

where $V$ contains eigenvectors (orthonormal directions) and $\Lambda$ contains eigenvalues (variance in each direction).

**Why this matters:** The covariance matrix in factor models decomposes into common and idiosyncratic components through eigenvalues and eigenvectors.

In [ ]:
# Purpose: Demonstrate eigendecomposition of a covariance matrix
# Key Concept: Eigenvalues measure variance, eigenvectors point in directions of maximum variance

def eigendecomposition_symmetric(A):
    """
    Compute eigendecomposition of symmetric matrix.
    
    Parameters
    ----------
    A : ndarray, shape (n, n)
        Symmetric matrix
    
    Returns
    -------
    eigenvalues : ndarray, shape (n,)
        Eigenvalues in descending order
    eigenvectors : ndarray, shape (n, n)
        Corresponding eigenvectors as columns
    """
    # Step 1: Use eigh for symmetric matrices (more efficient than eig)
    eigenvalues, eigenvectors = np.linalg.eigh(A)
    
    # Step 2: Sort in descending order (eigh returns ascending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    return eigenvalues, eigenvectors


# Create a sample covariance matrix
# Simulate data: 100 observations of 5 variables
X = np.random.randn(100, 5)
X[:, 0] = X[:, 0] + X[:, 1]  # Create correlation between variables
X[:, 2] = X[:, 2] + X[:, 3]
cov_matrix = np.cov(X.T)

# Compute eigendecomposition
eigenvalues, eigenvectors = eigendecomposition_symmetric(cov_matrix)

print("Covariance Matrix Shape:", cov_matrix.shape)
print("\nEigenvalues (sorted descending):")
print(eigenvalues.round(4))
print("\nSum of eigenvalues equals trace:")
print(f"  Sum(eigenvalues) = {eigenvalues.sum():.4f}")
print(f"  Trace(cov_matrix) = {np.trace(cov_matrix):.4f}")

# Verify reconstruction
A_reconstructed = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
reconstruction_error = np.linalg.norm(cov_matrix - A_reconstructed)
print(f"\nReconstruction error: {reconstruction_error:.2e} (should be ~0)")

### Visualizing Eigenvalues

The eigenvalue spectrum shows how variance is distributed across dimensions. In factor models, a few large eigenvalues indicate strong common factors.

In [ ]:
# Purpose: Visualize eigenvalue distribution to understand variance concentration
# Key Concept: Large eigenvalues = important directions of variation

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Eigenvalue magnitudes
axes[0].bar(range(1, len(eigenvalues) + 1), eigenvalues, alpha=0.7, color='steelblue')
axes[0].plot(range(1, len(eigenvalues) + 1), eigenvalues, 'ro-', linewidth=2)
axes[0].set_xlabel('Component Number')
axes[0].set_ylabel('Eigenvalue (Variance)')
axes[0].set_title('Eigenvalue Magnitudes (Scree Plot)')
axes[0].grid(True, alpha=0.3)

# Plot 2: Cumulative variance explained
cumulative_var = np.cumsum(eigenvalues) / eigenvalues.sum()
axes[1].plot(range(1, len(eigenvalues) + 1), cumulative_var, 'bo-', linewidth=2)
axes[1].axhline(y=0.9, color='r', linestyle='--', linewidth=2, label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Proportion of Variance')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Report variance concentration
n_components_90 = np.searchsorted(cumulative_var, 0.9) + 1
print(f"First eigenvalue explains {eigenvalues[0]/eigenvalues.sum()*100:.1f}% of variance")
print(f"Need {n_components_90} components to explain 90% of variance")

### Exercise 1.1: Eigendecomposition Properties

**Task:** Verify key properties of eigendecomposition for symmetric matrices.

**Expected Output:** All assertions should pass, confirming orthonormality and reconstruction.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Complete the verification function below
#
# Verify:
# 1. Eigenvectors are orthonormal: V'V = I
# 2. Reconstruction: A = V * Lambda * V'
# 3. Each column v_i satisfies: A * v_i = lambda_i * v_i

def verify_eigendecomposition(A, eigenvalues, eigenvectors):
    """
    Verify eigendecomposition properties.
    
    Returns
    -------
    dict with verification results
    """
    results = {}
    
    # 1. Check orthonormality: V'V should equal I
    VtV = None  # Replace with eigenvectors.T @ eigenvectors
    results['orthonormal'] = None  # Replace with np.allclose(VtV, np.eye(len(eigenvalues)))
    
    # 2. Check reconstruction: A = V * Lambda * V'
    A_reconstructed = None  # Replace with reconstruction formula
    results['reconstruction'] = None  # Replace with np.allclose(A, A_reconstructed)
    
    # 3. Check eigenvalue equation: A * v_i = lambda_i * v_i for first eigenvector
    v1 = eigenvectors[:, 0]
    Av1 = None  # Replace with A @ v1
    lambda_v1 = None  # Replace with eigenvalues[0] * v1
    results['eigenvalue_eq'] = None  # Replace with np.allclose(Av1, lambda_v1)
    
    return results

# Test your function
verification_results = verify_eigendecomposition(cov_matrix, eigenvalues, eigenvectors)
print("Verification Results:", verification_results)

<details>
<summary>Hint 1: Orthonormality</summary>
Orthonormal means V'V = I. Use @ operator for matrix multiplication and np.eye() for identity matrix.
</details>

<details>
<summary>Hint 2: Reconstruction</summary>
The reconstruction formula is: A = V @ np.diag(eigenvalues) @ V.T
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_1():
    results = verify_eigendecomposition(cov_matrix, eigenvalues, eigenvectors)
    
    assert results['orthonormal'] is not None, "Don't forget to compute orthonormality check!"
    assert results['reconstruction'] is not None, "Don't forget to compute reconstruction check!"
    assert results['eigenvalue_eq'] is not None, "Don't forget to compute eigenvalue equation check!"
    
    assert results['orthonormal'], "Eigenvectors should be orthonormal (V'V = I)"
    assert results['reconstruction'], "Reconstruction should match original matrix (A = V*Lambda*V')"
    assert results['eigenvalue_eq'], "Eigenvalue equation should hold (Av = lambda*v)"
    
    print("✓ Exercise 1.1 passed! All eigendecomposition properties verified.")

test_exercise_1_1()

In [ ]:
# SOLUTION
# --------
def verify_eigendecomposition_solution(A, eigenvalues, eigenvectors):
    results = {}
    
    # 1. Check orthonormality
    VtV = eigenvectors.T @ eigenvectors
    results['orthonormal'] = np.allclose(VtV, np.eye(len(eigenvalues)))
    
    # 2. Check reconstruction
    A_reconstructed = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
    results['reconstruction'] = np.allclose(A, A_reconstructed)
    
    # 3. Check eigenvalue equation
    v1 = eigenvectors[:, 0]
    Av1 = A @ v1
    lambda_v1 = eigenvalues[0] * v1
    results['eigenvalue_eq'] = np.allclose(Av1, lambda_v1)
    
    return results

## 1.2 Singular Value Decomposition (SVD)

For any matrix $X$ (not necessarily square or symmetric), SVD provides:
$$X = U \Sigma V'$$

where:
- $U$: Left singular vectors (directions in observation space)
- $\Sigma$: Singular values (importance/scaling)
- $V$: Right singular vectors (directions in variable space)

**Connection to PCA:** For centered data, PCA can be computed via SVD without forming the covariance matrix, which is more numerically stable and efficient.

In [ ]:
# Purpose: Implement SVD and show connection to eigendecomposition
# Key Concept: SVD generalizes eigendecomposition to non-square matrices

def svd_decomposition(X, n_components=None):
    """
    Compute SVD and optionally truncate to n_components.
    
    Parameters
    ----------
    X : ndarray, shape (T, N)
        Data matrix (observations x variables)
    n_components : int, optional
        Number of components to retain
    
    Returns
    -------
    U : ndarray, shape (T, r)
        Left singular vectors
    S : ndarray, shape (r,)
        Singular values
    Vt : ndarray, shape (r, N)
        Right singular vectors (transposed)
    """
    # Step 1: Compute full SVD (full_matrices=False for reduced SVD)
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    
    # Step 2: Optionally truncate to n_components
    if n_components is not None:
        U = U[:, :n_components]
        S = S[:n_components]
        Vt = Vt[:n_components, :]
    
    return U, S, Vt


# Generate sample data
T, N = 100, 5
X_raw = np.random.randn(T, N)
X_raw[:, 0] = X_raw[:, 0] + 2 * X_raw[:, 1]  # Create dependencies
X_raw[:, 2] = 0.5 * X_raw[:, 2] + X_raw[:, 3]

# Center the data (required for PCA interpretation)
X_centered = X_raw - X_raw.mean(axis=0)

# Compute SVD
U, S, Vt = svd_decomposition(X_centered)

print(f"Data shape: {X_centered.shape}")
print(f"U shape (left singular vectors): {U.shape}")
print(f"S shape (singular values): {S.shape}")
print(f"Vt shape (right singular vectors): {Vt.shape}")
print(f"\nSingular values:\n{S.round(4)}")

# Verify reconstruction
X_reconstructed = U @ np.diag(S) @ Vt
reconstruction_error = np.linalg.norm(X_centered - X_reconstructed, 'fro')
print(f"\nReconstruction error: {reconstruction_error:.2e}")

### Connection: SVD and Eigendecomposition

Key relationships:
- $X'X = V \Sigma^2 V'$ (eigendecomposition of Gram matrix)
- Singular values $\sigma_i$ relate to eigenvalues: $\lambda_i = \sigma_i^2 / T$

In [ ]:
# Purpose: Verify connection between SVD and eigendecomposition
# Key Concept: Singular values squared (scaled) equal eigenvalues of covariance

# Method 1: Eigendecomposition of covariance matrix
cov_X = np.cov(X_centered.T)
eigenvalues_cov, eigenvectors_cov = eigendecomposition_symmetric(cov_X)

# Method 2: From SVD singular values
eigenvalues_svd = S**2 / T

# Compare
print("Eigenvalues from covariance matrix:")
print(eigenvalues_cov.round(4))
print("\nEigenvalues from SVD (S^2 / T):")
print(eigenvalues_svd.round(4))
print("\nMatch:", np.allclose(eigenvalues_cov, eigenvalues_svd))

---

# Part 2: Principal Component Analysis (PCA)

PCA finds orthogonal directions that maximize variance. These directions become the basis for factor extraction in static factor models.

## 2.1 PCA from Scratch

**Objective:** Given data $X$, find directions $w$ that maximize projected variance:
$$\max_{\|w\|=1} \text{Var}(Xw)$$

**Solution:** Eigenvectors of the covariance matrix $\Sigma = \frac{1}{T}X'X$

In [ ]:
# Purpose: Implement PCA using two methods: covariance and SVD
# Key Concept: Both methods give same result; SVD is more efficient for large N

def pca_from_scratch(X, n_components=None, method='svd'):
    """
    Principal Component Analysis from scratch.
    
    Parameters
    ----------
    X : ndarray, shape (T, N)
        Data matrix (will be centered internally)
    n_components : int, optional
        Number of components to extract
    method : str
        'cov' for covariance eigendecomposition or 'svd' for SVD
    
    Returns
    -------
    scores : ndarray, shape (T, r)
        Principal component scores (factor estimates)
    loadings : ndarray, shape (N, r)
        Principal component loadings (factor loadings)
    eigenvalues : ndarray, shape (r,)
        Variance explained by each component
    """
    T, N = X.shape
    
    # Step 1: Center the data (subtract mean from each column)
    X_centered = X - X.mean(axis=0)
    
    if method == 'cov':
        # Method 1: Via covariance matrix eigendecomposition
        cov_matrix = X_centered.T @ X_centered / T
        eigenvalues, eigenvectors = eigendecomposition_symmetric(cov_matrix)
        
        # Loadings are eigenvectors
        loadings = eigenvectors
        
        # Scores are projections
        scores = X_centered @ loadings
        
    elif method == 'svd':
        # Method 2: Via SVD (more efficient, numerically stable)
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        
        # Eigenvalues from singular values
        eigenvalues = S**2 / T
        
        # Loadings are right singular vectors (columns of V)
        loadings = Vt.T
        
        # Scores are scaled left singular vectors
        scores = U * S  # Equivalent to X_centered @ loadings
    
    # Step 2: Truncate if requested
    if n_components is not None:
        scores = scores[:, :n_components]
        loadings = loadings[:, :n_components]
        eigenvalues = eigenvalues[:n_components]
    
    return scores, loadings, eigenvalues


# Test with sample data
scores_cov, loadings_cov, eigenvalues_cov = pca_from_scratch(X_raw, method='cov')
scores_svd, loadings_svd, eigenvalues_svd = pca_from_scratch(X_raw, method='svd')

print("PCA Results Comparison:")
print("\nEigenvalues match:", np.allclose(eigenvalues_cov, eigenvalues_svd))
print("Loadings match (up to sign):", np.allclose(np.abs(loadings_cov), np.abs(loadings_svd)))
print("Scores match (up to sign):", np.allclose(np.abs(scores_cov), np.abs(scores_svd)))

print("\nVariance explained by each PC:")
var_explained = eigenvalues_svd / eigenvalues_svd.sum()
for i, var in enumerate(var_explained, 1):
    print(f"  PC{i}: {var*100:.2f}%")

### Exercise 2.1: Implement Low-Rank Approximation

**Task:** Using PCA, create a rank-$r$ approximation of the data matrix that minimizes reconstruction error.

**Theory:** The optimal rank-$r$ approximation is $\hat{X} = \text{scores}_r \times \text{loadings}_r'$

**Expected Output:** Lower rank should have higher reconstruction error.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Compute reconstruction error for different numbers of components

def compute_reconstruction_error(X, n_components):
    """
    Compute reconstruction error for rank-n_components approximation.
    
    Parameters
    ----------
    X : ndarray, shape (T, N)
        Original data
    n_components : int
        Number of components for approximation
    
    Returns
    -------
    error : float
        Frobenius norm of reconstruction error
    """
    # Step 1: Run PCA with n_components
    scores, loadings, _ = None  # Replace with pca_from_scratch(X, n_components=n_components)
    
    # Step 2: Reconstruct data: X_hat = scores @ loadings.T
    X_centered = X - X.mean(axis=0)
    X_reconstructed = None  # Replace with reconstruction formula
    
    # Step 3: Compute Frobenius norm of error
    error = None  # Replace with np.linalg.norm(X_centered - X_reconstructed, 'fro')
    
    return error


# Test for different ranks
ranks = [1, 2, 3, 5]
errors = []

for r in ranks:
    error = compute_reconstruction_error(X_raw, r)
    errors.append(error)
    print(f"Rank {r}: reconstruction error = {error:.4f}")

<details>
<summary>Hint 1: PCA Call</summary>
Use: scores, loadings, _ = pca_from_scratch(X, n_components=n_components, method='svd')
</details>

<details>
<summary>Hint 2: Reconstruction</summary>
The reconstruction is: X_reconstructed = scores @ loadings.T
</details>

In [ ]:
# AUTO-GRADED TESTS
# ----------------------------------
def test_exercise_2_1():
    error_1 = compute_reconstruction_error(X_raw, 1)
    error_3 = compute_reconstruction_error(X_raw, 3)
    error_5 = compute_reconstruction_error(X_raw, 5)
    
    assert error_1 is not None, "Don't forget to compute reconstruction error!"
    assert error_1 > 0, "Reconstruction error should be positive"
    assert error_1 > error_3 > error_5, "Error should decrease as rank increases"
    assert error_5 < 1e-10, "Full rank (r=N) should have near-zero error"
    
    print("✓ Exercise 2.1 passed! Reconstruction errors computed correctly.")

test_exercise_2_1()

In [ ]:
# SOLUTION
# --------
def compute_reconstruction_error_solution(X, n_components):
    scores, loadings, _ = pca_from_scratch(X, n_components=n_components, method='svd')
    X_centered = X - X.mean(axis=0)
    X_reconstructed = scores @ loadings.T
    error = np.linalg.norm(X_centered - X_reconstructed, 'fro')
    return error

## 2.2 Interpreting PCA Results

**Loadings** tell us how each original variable contributes to each principal component.
**Scores** are the coordinates of observations in the principal component space.

In [ ]:
# Purpose: Create realistic macroeconomic-style data with interpretable structure
# Key Concept: Real-world factor structure with "real activity" and "inflation" factors

# Generate synthetic macro data with 2 factors
T = 200
np.random.seed(123)

# Factor 1: "Real Activity" - persistent AR(1) process
F1 = np.zeros(T)
F1[0] = np.random.randn()
for t in range(1, T):
    F1[t] = 0.8 * F1[t-1] + np.random.randn() * 0.6

# Factor 2: "Inflation" - less persistent
F2 = np.zeros(T)
F2[0] = np.random.randn()
for t in range(1, T):
    F2[t] = 0.5 * F2[t-1] + np.random.randn() * 0.8

F_true = np.column_stack([F1, F2])

# Create 6 variables with economic interpretation
# Variables 1-3: Load on real activity
# Variables 4-5: Load on inflation
# Variable 6: Loads on both
Lambda_true = np.array([
    [0.9, 0.1],   # Industrial production
    [0.85, 0.2],  # Employment
    [0.8, 0.15],  # Personal income
    [0.2, 0.9],   # CPI inflation
    [0.15, 0.85], # PPI inflation
    [0.6, 0.6],   # Interest rate (responds to both)
])

# Generate observed data
e = np.random.randn(T, 6) * 0.3  # Idiosyncratic noise
X_macro = F_true @ Lambda_true.T + e

# Variable names for interpretation
var_names = ['IP', 'Employment', 'Income', 'CPI', 'PPI', 'IntRate']

print("Generated macroeconomic panel:")
print(f"  Time periods: {T}")
print(f"  Variables: {len(var_names)}")
print(f"  True factors: 2 (Real Activity, Inflation)")

In [ ]:
# Purpose: Extract and visualize PCA factors
# Key Concept: PC loadings reveal economic interpretation

# Run PCA
scores, loadings, eigenvalues = pca_from_scratch(X_macro, n_components=2)

# Visualize loadings
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PC1 loadings
axes[0].barh(var_names, loadings[:, 0], alpha=0.7, color='steelblue')
axes[0].set_xlabel('Loading on PC1')
axes[0].set_title('First Principal Component Loadings')
axes[0].axvline(x=0, color='black', linewidth=0.5)
axes[0].grid(True, alpha=0.3)

# PC2 loadings
axes[1].barh(var_names, loadings[:, 1], alpha=0.7, color='coral')
axes[1].set_xlabel('Loading on PC2')
axes[1].set_title('Second Principal Component Loadings')
axes[1].axvline(x=0, color='black', linewidth=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Variance explained
var_explained = eigenvalues / eigenvalues.sum()
print(f"\nPC1 explains {var_explained[0]*100:.1f}% of variance")
print(f"PC2 explains {var_explained[1]*100:.1f}% of variance")
print(f"Total: {var_explained.sum()*100:.1f}%")

**Interpretation Note:** 
- PC1 likely captures **real activity** (high loadings on IP, Employment, Income)
- PC2 likely captures **inflation** (high loadings on CPI, PPI)
- Interest rate loads on both (responds to economic activity and inflation)

This demonstrates how PCA discovers latent factors from observed co-movement.

In [ ]:
callout("**", "warning")

---

# Part 3: Time Series Basics

Dynamic factor models extend static factors by adding time series dynamics. We need to understand stationarity and autocovariance.

## 3.1 Stationarity

A time series is **weakly stationary** if:
1. $E[y_t] = \mu$ (constant mean)
2. $\text{Var}(y_t) = \sigma^2$ (constant variance)
3. $\text{Cov}(y_t, y_{t-h}) = \gamma(h)$ (autocovariance depends only on lag $h$)

In [ ]:
# Purpose: Simulate and visualize stationary vs non-stationary processes
# Key Concept: Stationary processes have stable statistical properties

def simulate_ar1(phi, sigma=1.0, T=200, y0=0):
    """
    Simulate AR(1) process: y_t = phi * y_{t-1} + eps_t
    
    Parameters
    ----------
    phi : float
        Autoregressive parameter (|phi| < 1 for stationarity)
    sigma : float
        Standard deviation of innovations
    T : int
        Number of time periods
    y0 : float
        Initial value
    
    Returns
    -------
    y : ndarray, shape (T,)
        Simulated time series
    """
    y = np.zeros(T)
    y[0] = y0
    
    for t in range(1, T):
        y[t] = phi * y[t-1] + np.random.randn() * sigma
    
    return y


# Simulate three processes
T = 300
np.random.seed(456)

y_stationary = simulate_ar1(phi=0.7, T=T)       # Stationary: |phi| < 1
y_near_unit = simulate_ar1(phi=0.98, T=T)       # Near unit root: very persistent
y_random_walk = simulate_ar1(phi=1.0, T=T)      # Non-stationary: random walk

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(12, 9))

axes[0].plot(y_stationary, linewidth=1.5, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Stationary AR(1): φ=0.7')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_near_unit, linewidth=1.5, color='darkgreen')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Near Unit Root AR(1): φ=0.98')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3)

axes[2].plot(y_random_walk, linewidth=1.5, color='darkred')
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('Random Walk (Non-stationary): φ=1.0')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Value')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare sample statistics
print("Sample Statistics:")
print(f"\nStationary (φ=0.7):")
print(f"  Mean: {y_stationary.mean():.3f}")
print(f"  Std: {y_stationary.std():.3f}")
print(f"\nNear Unit Root (φ=0.98):")
print(f"  Mean: {y_near_unit.mean():.3f}")
print(f"  Std: {y_near_unit.std():.3f}")
print(f"\nRandom Walk (φ=1.0):")
print(f"  Mean: {y_random_walk.mean():.3f}")
print(f"  Std: {y_random_walk.std():.3f} (grows with T!)")

### Exercise 3.1: Compute Autocovariance Function

**Task:** Compute and plot the autocovariance function for a stationary AR(1) process.

**Theory:** For AR(1) with $|\phi| < 1$, the autocovariance at lag $h$ is:
$$\gamma(h) = \frac{\sigma^2}{1-\phi^2} \cdot \phi^h$$

**Expected Output:** Sample autocovariance should match theoretical formula.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Compute sample autocovariance for different lags

def compute_autocovariance(y, max_lag=10):
    """
    Compute sample autocovariance function.
    
    Parameters
    ----------
    y : ndarray, shape (T,)
        Time series data
    max_lag : int
        Maximum lag to compute
    
    Returns
    -------
    acov : ndarray, shape (max_lag+1,)
        Autocovariance at lags 0, 1, ..., max_lag
    """
    T = len(y)
    y_centered = y - y.mean()
    acov = np.zeros(max_lag + 1)
    
    for h in range(max_lag + 1):
        # Compute Cov(y_t, y_{t-h}) = E[(y_t - mu)(y_{t-h} - mu)]
        if h == 0:
            acov[h] = None  # Replace with np.mean(y_centered**2)
        else:
            acov[h] = None  # Replace with np.mean(y_centered[h:] * y_centered[:-h])
    
    return acov


# Compute for stationary process
acov_sample = compute_autocovariance(y_stationary, max_lag=20)

# Theoretical autocovariance for AR(1)
phi = 0.7
sigma = 1.0
gamma_0 = sigma**2 / (1 - phi**2)
lags = np.arange(21)
acov_theoretical = gamma_0 * phi**lags

# Plot comparison
plt.figure(figsize=(10, 5))
plt.plot(lags, acov_theoretical, 'r-', linewidth=2, label='Theoretical')
plt.plot(lags, acov_sample, 'bo--', linewidth=1.5, label='Sample', markersize=6)
plt.xlabel('Lag')
plt.ylabel('Autocovariance')
plt.title('Autocovariance Function: AR(1) with φ=0.7')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary>Hint 1: Lag 0</summary>
At lag 0, autocovariance equals variance: acov[0] = np.mean(y_centered**2)
</details>

<details>
<summary>Hint 2: Lag h > 0</summary>
For lag h, compute: acov[h] = np.mean(y_centered[h:] * y_centered[:-h])
This multiplies y_t and y_{t-h} and takes the mean.
</details>

In [ ]:
# AUTO-GRADED TESTS
# ----------------------------------
def test_exercise_3_1():
    acov = compute_autocovariance(y_stationary, max_lag=10)
    
    assert acov is not None, "Don't forget to compute autocovariance!"
    assert len(acov) == 11, "Should return autocovariance for lags 0 to 10"
    assert acov[0] > 0, "Lag 0 autocovariance (variance) should be positive"
    assert acov[0] > acov[1] > acov[2], "Autocovariance should decay for AR(1)"
    
    # Check approximate match with theoretical values
    gamma_0 = 1.0 / (1 - 0.7**2)
    theoretical_lag1 = gamma_0 * 0.7
    assert np.abs(acov[1] - theoretical_lag1) < 0.5, "Sample autocov should be close to theoretical"
    
    print("✓ Exercise 3.1 passed! Autocovariance computed correctly.")

test_exercise_3_1()

In [ ]:
# SOLUTION
# --------
def compute_autocovariance_solution(y, max_lag=10):
    T = len(y)
    y_centered = y - y.mean()
    acov = np.zeros(max_lag + 1)
    
    for h in range(max_lag + 1):
        if h == 0:
            acov[h] = np.mean(y_centered**2)
        else:
            acov[h] = np.mean(y_centered[h:] * y_centered[:-h])
    
    return acov

---

# Summary and Key Takeaways

## What You've Learned

1. **Matrix Decompositions**
   - Eigendecomposition finds natural axes of covariance matrices
   - SVD generalizes eigendecomposition to rectangular matrices
   - Both are essential for factor extraction

2. **Principal Component Analysis**
   - PCA finds directions of maximum variance
   - Loadings reveal how variables contribute to components
   - Scores are coordinates in the reduced-dimensional space
   - PCA forms the basis for static factor model estimation

3. **Time Series Concepts**
   - Stationarity requires constant mean, variance, and autocovariance structure
   - AR(1) processes demonstrate persistence and mean reversion
   - Autocovariance measures temporal dependence
   - Dynamic factor models extend factors with time series dynamics

## Connection to Factor Models

Everything in this notebook connects directly to factor models:

| Foundation | Application |
|------------|-------------|
| Eigendecomposition | Extract factors via PCA |
| SVD | Efficient computation for large panels |
| PCA loadings | Become factor loadings in static models |
| PCA scores | Become factor estimates |
| Stationarity | Required for consistent factor estimation |
| Autocovariance | Specifies factor dynamics in DFMs |

## Next Steps

You're now ready to:
1. Proceed to **Module 1: Static Factor Models** - Learn the formal factor model specification
2. See how PCA estimates factors in the Stock-Watson framework
3. Understand identification issues and normalization constraints

---

## Self-Assessment

Before moving on, ensure you can:
- [ ] Compute eigendecomposition and explain what eigenvalues/eigenvectors represent
- [ ] Implement PCA from scratch using both covariance and SVD methods
- [ ] Interpret PCA loadings in terms of variable contributions
- [ ] Distinguish between stationary and non-stationary processes
- [ ] Compute and interpret autocovariance functions

If any checkbox is unclear, review that section before continuing.

## Additional Resources

- **Matrix Algebra:** Strang, *Introduction to Linear Algebra*, Chapters 6-7
- **PCA:** Jolliffe, *Principal Component Analysis*, 2nd ed.
- **Time Series:** Hamilton, *Time Series Analysis*, Chapters 2-3
- **Online:** Matrix Cookbook (matrixcookbook.com)

---

*Congratulations on completing the foundations review! You now have the mathematical tools needed for dynamic factor models.*